# ⚡ AETHER All-in-One Studio — Google Colab & Kaggle Runner

**Models:** Stable Diffusion XL (Images) + Fish Audio S2 Pro (Voice)

This notebook runs **BOTH** models simultaneously on a single free T4 GPU using a unified API tunnel and CPU offloading!

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Install Dependencies
%cd /content
!pip install -q diffusers accelerate torch fastapi uvicorn httpx pyngrok nest_asyncio
print("✅ Dependencies installed!")


In [ ]:
# 2. Authenticate ngrok
# REPLACE "YOUR_TOKEN_HERE" WITH YOUR ACTUAL NGROK TOKEN IF SECRETS ARE NOT WORKING
MANUAL_TOKEN = ""

try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

try:
    if MANUAL_TOKEN:
        ngrok_token = MANUAL_TOKEN
    elif colab_env:
        ngrok_token = userdata.get("NGROK_TOKEN")
    else:
        # For Kaggle or other envs without Colab userdata
        import os
        ngrok_token = os.environ.get("NGROK_TOKEN", "")
        
    if not ngrok_token:
        raise ValueError("Token is empty!")
        
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print("\n❌ FATAL ERROR: Could not authenticate with ngrok!")
    print("You have two options to fix this:")
    print("1. Paste your token between the quotes in MANUAL_TOKEN = \"\" at the top of this cell.")
    print("2. OR Add your ngrok token to Colab Secrets (the 🔑 icon on the left) as NGROK_TOKEN and turn the toggle switch ON.")
    raise Exception("STOPPING: You must provide a valid ngrok token before continuing!")

In [ ]:
# 3. Launch Unified API Server
import nest_asyncio
import uvicorn
import base64
import torch
import httpx
import subprocess
import time
import requests
import os
import psutil
from io import BytesIO
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from diffusers import StableDiffusionXLPipeline

nest_asyncio.apply()

# Ensure any zombie ngrok tunnels from previous interrupted runs are killed
ngrok.kill()
os.system("killall -9 ngrok 2>/dev/null")
for conn in psutil.net_connections():
    if conn.laddr.port in [8000, 8081] and conn.status == 'LISTEN':
        try:
            psutil.Process(conn.pid).terminate()
        except:
            pass
time.sleep(1)


print("\n🎨 Loading Stable Diffusion XL Base 1.0 (with CPU Offload)...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
pipe.to("cuda")

pipe.enable_attention_slicing()
print("✅ SDXL Ready!")

app = FastAPI(title="AETHER All-in-One")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/v1/health")
def health():
    return {"status": "ok"}

@app.get("/api/health")
def api_health():
    return {"status": "ok"}

class GenerateReq(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, low quality"
    width: int = 1024
    height: int = 1024
    steps: int = 30
    guidance_scale: float = 7.0
    seed: int = -1

@app.post("/generate")
def generate(req: GenerateReq):
    try:
        seed = req.seed if req.seed != -1 else torch.randint(0, 2**32, (1,)).item()
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
        result = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            width=req.width,
            height=req.height,
            num_inference_steps=req.steps,
            guidance_scale=req.guidance_scale,
            generator=generator,
        )
        buffer = BytesIO()
        result.images[0].save(buffer, format="PNG")
        b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        torch.cuda.empty_cache()
        return {"image_base64": b64}
    except Exception as e:
        import traceback
        return {"error": str(e), "traceback": traceback.format_exc()}


# --- img2img: render a scene FROM a reference image -------------------------
#
# Lets AetherStudio carry a character's reference portrait into their scenes
# rather than relying on the seed alone to keep a face recognisable.
#
# from_pipe() reuses the UNet, VAE and text encoders already loaded above, so
# this costs NO extra VRAM and no second download — which matters on a T4 where
# SDXL already occupies most of the card.
#
# The app discovers this by reading /openapi.json and matching the route name,
# so nothing needs changing on its side once this cell has run.
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image
import io

img2img_pipe = StableDiffusionXLImg2ImgPipeline.from_pipe(pipe)
img2img_pipe.to("cuda")


class Img2ImgReq(BaseModel):
    prompt: str
    # Both spellings are accepted: different callers name this differently and
    # a silently ignored reference image is worse than a clear error.
    image: str = ""
    init_image: str = ""
    negative_prompt: str = "blurry, low quality"
    # How far to travel from the reference. ~0.3 stays very close (and keeps
    # the reference's framing), ~0.85 is nearly a fresh image with the identity
    # mostly lost. 0.65 holds identity while allowing a new composition.
    strength: float = 0.65
    denoising_strength: float = -1.0
    steps: int = 30
    guidance_scale: float = 7.0
    seed: int = -1


def _decode_b64_image(raw: str):
    if raw.strip().startswith("data:") and "," in raw:
        raw = raw.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(raw))).convert("RGB")


@app.post("/img2img")
def img2img(req: Img2ImgReq):
    try:
        raw = req.image or req.init_image
        if not raw:
            return {"error": "No image supplied. Send base64 (or a data URL) as 'image' or 'init_image'."}

        init = _decode_b64_image(raw)
        # SDXL needs dimensions divisible by 8. A reference portrait is rarely
        # the scene's aspect ratio, so resize instead of failing.
        w = max(8, (init.width // 8) * 8)
        h = max(8, (init.height // 8) * 8)
        if (w, h) != init.size:
            init = init.resize((w, h))

        strength = req.denoising_strength if req.denoising_strength >= 0 else req.strength
        strength = max(0.05, min(1.0, float(strength)))

        seed = req.seed if req.seed != -1 else torch.randint(0, 2**32, (1,)).item()
        generator = torch.Generator(device="cuda").manual_seed(int(seed))

        result = img2img_pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            image=init,
            strength=strength,
            num_inference_steps=req.steps,
            guidance_scale=req.guidance_scale,
            generator=generator,
        )

        buffer = BytesIO()
        result.images[0].save(buffer, format="PNG")
        b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        torch.cuda.empty_cache()
        # Mirrors /generate's shape (image_base64) and also supplies an images
        # array, so any caller expecting either form works unchanged.
        return {"image_base64": b64, "images": [{"base64": b64, "seed": int(seed)}], "seed": int(seed)}

    except Exception as e:
        import traceback
        torch.cuda.empty_cache()
        return {"error": str(e), "traceback": traceback.format_exc()}


print("✅ /img2img ready — shares the loaded SDXL pipeline, no extra VRAM.")


public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print("🚀 AETHER ALL-IN-ONE API IS LIVE  (txt2img + img2img)")
print("="*60)
print(f"  Paste this ONE link into the STABLE DIFFUSION Settings box in Blvck-TTS:")
print(f"  URL: {public_url}")
print("="*60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
import asyncio
await server.serve()
